# Notebook 5 — Build Tools and the UFC AI Agent

This notebook connects the data and vector store created by Notebooks 2–4, then builds a tool-using UFC analyst agent and a Gradio chat interface.

**Project folder in Google Drive:** `/content/drive/MyDrive/RAG UFC-AI-Analyst`

## Before running
1. Run Notebook 2, Notebook 3, and Notebook 4 successfully first.
2. Confirm `data/ufc.duckdb`, `documents/`, and `vectorstore/chroma_db/` exist in this project folder.
3. You need a Groq API key. The notebook asks for it securely; do not paste the key into a notebook cell or commit it to GitHub.
4. Run the cells from top to bottom. The Gradio cell at the end starts the chat app and may provide a temporary public link.


In [8]:
%pip install -q -U duckdb chromadb sentence-transformers langchain langchain-community langchain-huggingface langchain-chroma langchain-groq langgraph gradio

In [9]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# Cell 3 — Configure and verify project paths
from pathlib import Path
import os
import json

PROJECT_ROOT = Path("/content/drive/MyDrive/RAG UFC-AI-Analyst")
DB_PATH = PROJECT_ROOT / "data" / "ufc.duckdb"
DOCUMENTS_DIR = PROJECT_ROOT / "documents"
VECTORSTORE_DIR = PROJECT_ROOT / "vectorstore" / "chroma_db"
COLLECTION_NAME = "ufc_rag"

print("Project:", PROJECT_ROOT, "| exists:", PROJECT_ROOT.is_dir())
print("DuckDB:", DB_PATH, "| exists:", DB_PATH.is_file())
print("Documents:", DOCUMENTS_DIR, "| exists:", DOCUMENTS_DIR.is_dir())
print("ChromaDB:", VECTORSTORE_DIR, "| exists:", VECTORSTORE_DIR.is_dir())

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Project folder not found: {PROJECT_ROOT}. Check the exact folder name in MyDrive.")
if not DB_PATH.is_file():
    raise FileNotFoundError(f"DuckDB database not found: {DB_PATH}. Run Notebook 2 first.")
if not DOCUMENTS_DIR.is_dir():
    raise FileNotFoundError(f"Documents folder not found: {DOCUMENTS_DIR}. Run Notebook 3 first.")
if not VECTORSTORE_DIR.is_dir():
    raise FileNotFoundError(f"ChromaDB folder not found: {VECTORSTORE_DIR}. Run Notebook 4 first.")


Project: /content/drive/MyDrive/RAG UFC-AI-Analyst | exists: True
DuckDB: /content/drive/MyDrive/RAG UFC-AI-Analyst/data/ufc.duckdb | exists: True
Documents: /content/drive/MyDrive/RAG UFC-AI-Analyst/documents | exists: True
ChromaDB: /content/drive/MyDrive/RAG UFC-AI-Analyst/vectorstore/chroma_db | exists: True


In [11]:
# Cell 4 — Open DuckDB read-only and inspect the schema
import duckdb
import pandas as pd

# Read-only avoids writing to the database from this agent notebook.
conn = duckdb.connect(str(DB_PATH), read_only=True)

tables_df = conn.execute("SHOW TABLES").fetchdf()
print("Tables:", tables_df["name"].tolist())

for table_name in tables_df["name"].tolist():
    columns = conn.execute(f"DESCRIBE {table_name}").fetchdf()["column_name"].tolist()
    row_count = conn.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]
    print(f"{table_name}: {row_count:,} rows")
    print("  Columns:", ", ".join(columns))


Tables: ['events', 'fight_performances', 'fighters', 'fights']
events: 794 rows
  Columns: event_id, event_name, event_date, event_location
fight_performances: 17,784 rows
  Columns: fight_id, fighter_id, opponent_id, age, weight, sig_strikes_landed, sig_strikes_attempted, total_strikes_landed, total_strikes_attempted, body_strikes_landed, body_strikes_attempted, clinch_strikes_landed, clinch_strikes_attempted, distance_strikes_landed, distance_strikes_attempted, ground_strikes_landed, ground_strikes_attempted, head_strikes_landed, head_strikes_attempted, leg_strikes_landed, leg_strikes_attempted, takedowns_landed, takedowns_attempted, submission_attempts, knockdowns, reversals, control_time_per_fight, round1_sig_strikes_landed, round1_sig_strikes_attempted, round1_control_time, round1_reversals, round1_takedowns_landed, round1_takedowns_attempted, round1_knockdowns, round1_submission_attempts, round1_head_strikes_landed, round1_head_strikes_attempted, round1_body_strikes_landed, round

In [12]:
# Cell 5 — Load the same embedding model used in Notebook 4
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Must match Notebook 4's all-MiniLM-L6-v2 model.
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": False},
)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=str(VECTORSTORE_DIR),
    embedding_function=embedding_model,
)

vector_count = vectorstore._collection.count()
print("Chroma collection:", COLLECTION_NAME)
print("Stored chunks:", vector_count)

if vector_count == 0:
    raise ValueError("The Chroma collection is empty. Run Notebook 4 successfully first.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chroma collection: ufc_rag
Stored chunks: 11472


In [13]:
# Cell 6 — Tool 1: semantic document retrieval
from langchain.tools import tool

@tool
def retrieve_documents(query: str, k: int = 5) -> str:
    """Search UFC fighter, fight, and event documents for relevant background or descriptions.

    Use this for questions asking what happened, event summaries, or contextual details.
    For exact counts, wins, losses, and statistics, use the SQL tools instead.
    Args:
        query: A natural-language search query.
        k: Maximum number of passages to retrieve (1–10).
    """
    k = max(1, min(int(k), 10))
    docs = vectorstore.similarity_search(query, k=k)
    if not docs:
        return "No relevant documents found."

    results = []
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "unknown")
        category = doc.metadata.get("category", "unknown")
        results.append(f"Result {i} | category={category} | source={source}\n{doc.page_content}")
    return "\n\n---\n\n".join(results)

print("Created:", retrieve_documents.name)


Created: retrieve_documents


In [14]:
# Cell 7 — Tool 2: run read-only SQL queries
@tool
def run_sql_query(sql: str) -> str:
    """Run a read-only SELECT query against the UFC DuckDB database.

    Available tables:
    - fighters: fighter_id, name, dob, height_cm, reach_cm
    - events: event_id, name, date, location
    - fights: fight_id, event_id, weight_class, method, round, time, bonus, bout_type
    - fight_performances: fight_id, fighter_id, opponent_id, is_winner and fight statistics

    Important: each fight_performances row describes that fighter's own performance.
    For defensive statistics against a fighter, inspect the opponent's row.
    Date is stored in events, not fights. For recent fights, rank all wins and losses by event date.
    Only one SELECT or WITH...SELECT statement is allowed.
    """
    cleaned = sql.strip()
    lowered = cleaned.lower()

    # Basic safety checks: only a single read query is allowed.
    if ";" in cleaned.rstrip(";"):
        return "SQL Error: Please provide exactly one SQL statement."
    if not (lowered.startswith("select") or lowered.startswith("with")):
        return "SQL Error: Only SELECT or WITH...SELECT queries are allowed."
    forbidden = [" insert ", " update ", " delete ", " drop ", " alter ", " create ", " attach ", " copy ", " export ", " install ", " load ", " pragma ", " call "]
    padded = f" {lowered} "
    if any(word in padded for word in forbidden):
        return "SQL Error: The query contains a disallowed operation."

    try:
        result = conn.execute(cleaned).fetchdf()
        if result.empty:
            return "Query returned no rows."
        return result.head(100).to_string(index=False)
    except Exception as exc:
        return f"SQL Error: {exc}"

print("Created:", run_sql_query.name)


Created: run_sql_query


In [15]:
# Cell 8 — Tool 3: look up a matchup
@tool
def get_fight(fighter_a: str, fighter_b: str) -> str:
    """Find recorded UFC fights between two fighters and return event, result, method, and stats."""
    sql = """
    SELECT
        e.name AS event,
        e.date,
        e.location,
        winner.name AS winner,
        loser.name AS loser,
        fi.method,
        fi.round,
        fi.time,
        fi.weight_class,
        fi.bonus,
        fp_w.sig_strikes_landed AS winner_sig_strikes,
        fp_l.sig_strikes_landed AS loser_sig_strikes,
        fp_w.knockdowns AS winner_knockdowns,
        fp_l.knockdowns AS loser_knockdowns,
        fp_w.takedowns_landed AS winner_takedowns,
        fp_l.takedowns_landed AS loser_takedowns,
        fp_w.submission_attempts AS winner_sub_attempts,
        fp_l.submission_attempts AS loser_sub_attempts
    FROM fights fi
    JOIN events e ON fi.event_id = e.event_id
    JOIN fight_performances fp_w
        ON fi.fight_id = fp_w.fight_id AND fp_w.is_winner = TRUE
    JOIN fighters winner ON fp_w.fighter_id = winner.fighter_id
    JOIN fight_performances fp_l
        ON fi.fight_id = fp_l.fight_id AND fp_l.is_winner = FALSE
    JOIN fighters loser ON fp_l.fighter_id = loser.fighter_id
    WHERE
        (winner.name ILIKE ? AND loser.name ILIKE ?)
        OR (winner.name ILIKE ? AND loser.name ILIKE ?)
    ORDER BY e.date DESC
    LIMIT 10
    """
    try:
        a = f"%{fighter_a}%"
        b = f"%{fighter_b}%"
        result = conn.execute(sql, [a, b, b, a]).fetchdf()
        if result.empty:
            return f"No recorded fight found matching '{fighter_a}' and '{fighter_b}'. Check the spelling or try shorter names."
        return result.to_string(index=False)
    except Exception as exc:
        return f"Fight lookup error: {exc}"

print("Created:", get_fight.name)


Created: get_fight


In [16]:
# Cell 9 — Tool 4: fighter career summary and recent fights
@tool
def get_fighter(fighter_name: str) -> str:
    """Return a fighter's recorded career totals and their five most recent recorded fights."""
    career_sql = """
    SELECT
        f.name,
        COUNT(*) AS recorded_fights,
        SUM(CASE WHEN fp.is_winner THEN 1 ELSE 0 END) AS wins,
        SUM(CASE WHEN NOT fp.is_winner THEN 1 ELSE 0 END) AS losses,
        SUM(fp.sig_strikes_landed) AS total_sig_strikes_landed,
        SUM(fp.takedowns_landed) AS total_takedowns_landed,
        SUM(fp.knockdowns) AS total_knockdowns
    FROM fighters f
    JOIN fight_performances fp ON f.fighter_id = fp.fighter_id
    WHERE f.name ILIKE ?
    GROUP BY f.name
    ORDER BY recorded_fights DESC
    LIMIT 10
    """
    recent_sql = """
    SELECT
        e.date,
        e.name AS event,
        CASE WHEN fp.is_winner THEN 'Win' ELSE 'Loss' END AS result,
        opp.name AS opponent,
        fi.method,
        fi.round,
        fi.time
    FROM fight_performances fp
    JOIN fighters f ON fp.fighter_id = f.fighter_id
    JOIN fighters opp ON fp.opponent_id = opp.fighter_id
    JOIN fights fi ON fp.fight_id = fi.fight_id
    JOIN events e ON fi.event_id = e.event_id
    WHERE f.name ILIKE ?
    ORDER BY e.date DESC
    LIMIT 5
    """
    try:
        pattern = f"%{fighter_name}%"
        career = conn.execute(career_sql, [pattern]).fetchdf()
        if career.empty:
            return f"No fighter found matching '{fighter_name}'. Try a shorter or differently spelled name."
        recent = conn.execute(recent_sql, [pattern]).fetchdf()
        return (
            "=== Career Summary (recorded dataset) ===\n"
            + career.to_string(index=False)
            + "\n\n=== Most Recent Recorded Fights ===\n"
            + (recent.to_string(index=False) if not recent.empty else "No recent fights found.")
        )
    except Exception as exc:
        return f"Fighter lookup error: {exc}"

print("Created:", get_fighter.name)


Created: get_fighter


In [17]:
# Cell 10 — Tool 5: event fight card and results
@tool
def get_event(event_name: str) -> str:
    """Return the recorded fight card and results for a UFC event name."""
    sql = """
    SELECT
        e.name AS event,
        e.date,
        e.location,
        winner.name AS winner,
        loser.name AS loser,
        fi.weight_class,
        fi.method,
        fi.round,
        fi.time,
        fi.bonus
    FROM events e
    JOIN fights fi ON fi.event_id = e.event_id
    JOIN fight_performances fp_w
        ON fi.fight_id = fp_w.fight_id AND fp_w.is_winner = TRUE
    JOIN fighters winner ON fp_w.fighter_id = winner.fighter_id
    JOIN fight_performances fp_l
        ON fi.fight_id = fp_l.fight_id AND fp_l.is_winner = FALSE
    JOIN fighters loser ON fp_l.fighter_id = loser.fighter_id
    WHERE e.name ILIKE ?
    ORDER BY fi.weight_class, winner.name
    LIMIT 100
    """
    try:
        result = conn.execute(sql, [f"%{event_name}%"]).fetchdf()
        if result.empty:
            return f"No event found matching '{event_name}'. Try a shorter event name such as 'UFC 300'."
        return result.to_string(index=False)
    except Exception as exc:
        return f"Event lookup error: {exc}"

print("Created:", get_event.name)


Created: get_event


In [18]:
# Cell 11 — Register tools and verify each tool is available
tools = [
    retrieve_documents,
    run_sql_query,
    get_fight,
    get_fighter,
    get_event,
]
print("Registered tools:")
for item in tools:
    print("-", item.name)


Registered tools:
- retrieve_documents
- run_sql_query
- get_fight
- get_fighter
- get_event


In [19]:
# Cell 12 — Enter your Groq API key securely
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key (input hidden): ")

if not os.environ.get("GROQ_API_KEY", "").strip():
    raise ValueError("GROQ_API_KEY is empty. Re-run this cell and enter your key.")

print("Groq API key is set for this Colab runtime. It is not printed.")


Enter your Groq API key (input hidden): ··········
Groq API key is set for this Colab runtime. It is not printed.


In [20]:
# Cell 13 — Create the language model
from langchain_groq import ChatGroq
from langchain.agents import create_agent

MODEL_NAME = "openai/gpt-oss-120b"

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0,
    max_tokens=2048,
)

print("Configured model:", MODEL_NAME)


Configured model: openai/gpt-oss-120b


In [21]:
# Cell 14 — Create the tool-using UFC AI Analyst agent
system_prompt = """You are a careful UFC AI Analyst.

Rules:
1. Use the tools for factual claims. Do not invent facts.
2. If tool results do not contain the answer, say that the available dataset does not establish it.
3. Prefer structured SQL tools for exact results, methods, rounds, dates, and statistics.
4. Use retrieve_documents for contextual text, not as the authority when it conflicts with structured data.
5. The database reflects the records available in this project and may not include every current or historical UFC event.
6. Distinguish a fighter's own offensive statistics from statistics landed by opponents against them.
7. Explain the relevant dataset scope when giving career totals; do not claim the data is complete unless verified.
8. Be concise, and show calculations or SQL-derived rows when helpful.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
)
print("UFC AI Analyst agent created.")


UFC AI Analyst agent created.


In [22]:
# Cell 15 — Test the tools and ask a first question
print("Chroma retrieval test:")
print(retrieve_documents.invoke({"query": "UFC fighter profile and career summary", "k": 2})[:1200])

print("\\nFighter tool test:")
print(get_fighter.invoke({"fighter_name": "Khamzat Chimaev"})[:1800])

def ask(question: str):
    """Ask the agent a question and return its final response."""
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    final_message = result["messages"][-1]
    answer = final_message.content
    if isinstance(answer, list):
        answer = "\\n".join(
            part.get("text", str(part)) if isinstance(part, dict) else str(part)
            for part in answer
        )
    print(answer)
    return result

# Example question. If you prefer, replace it with your own UFC question.
ask("Who won Pereira vs Hill at UFC 300, and what method was recorded?")


Chroma retrieval test:
Result 1 | category=fighters | source=fighters/BRIAN_GERAGHTY_15edcf67ccf5be84.txt
# UFC Fighter Profile: BRIAN GERAGHTY
Fighter ID: 15edcf67ccf5be84
Date of birth: 1980-11-04 00:00:00
Height: 67.0
Reach: Unknown

## Career summary
Recorded fights in this dataset: 1
Wins based on derived source labels: 0
Losses based on derived source labels: 1
Fights with unknown result: 0

## Recent recorded fights
- Fight ID 62454fb3a4fd5737: loss vs MATT WIMAN; event: The Ultimate Fighter: Team Pulver vs. Team Penn Finale; date: 2007-06-23 00:00:00; weight class: Lightweight.

---

Result 2 | category=fighters | source=fighters/SATOSHI_HONMA_d2b1c1317a39f6c6.txt
# UFC Fighter Profile: SATOSHI HONMA
Fighter ID: d2b1c1317a39f6c6
Date of birth: 1968-05-19 00:00:00
Height: 73.0
Reach: Unknown

## Career summary
Recorded fights in this dataset: 1
Wins based on derived source labels: 0
Losses based on derived source labels: 1
Fights with unknown result: 0

## Recent recorded fights

{'messages': [HumanMessage(content='Who won Pereira vs Hill at UFC 300, and what method was recorded?', additional_kwargs={}, response_metadata={}, id='4246cb63-a21a-4c17-a1fb-3ca4e8111754'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer: Who won Pereira vs Hill at UFC 300, and what method was recorded? Use get_fight perhaps. Let\'s call get_fight with fighter_a "Alex Pereira" and fighter_b "Jared Cannonier"? Wait Hill? Actually "Hill" likely "Julianna Hill"? No, Pereira vs Hill: Could be "Alex Pereira vs. Sean Strickland"? No. There\'s a fighter named "Julianna Hill". But Pereira? There\'s "Alex Pereira" (kickboxer) vs "Julianna Hill"? That seems unlikely. Could be "Mackenzie Hill"? Not sure. Let\'s search documents.', 'tool_calls': [{'id': 'fc_778eb40d-4739-4342-a73f-0c346ce2cf71', 'function': {'arguments': '{"k":5,"query":"Pereira vs Hill UFC 300"}', 'name': 'retrieve_documents'}, 'type': 'function'}]}, response_metadata={'token_usage': {'complet

In [23]:
# Cell 16 — Launch the Gradio chat interface
import gradio as gr

def chat(message, history):
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": message}]})
        final_message = result["messages"][-1]
        answer = final_message.content
        if isinstance(answer, list):
            answer = "\\n".join(
                part.get("text", str(part)) if isinstance(part, dict) else str(part)
                for part in answer
            )
        return str(answer)
    except Exception as exc:
        return f"An error occurred: {exc}"

demo = gr.ChatInterface(
    fn=chat,
    title="UFC AI Analyst",
    description="Ask questions about recorded UFC fights, fighters, events, and statistics.",
    examples=[
        "Who won Pereira vs Hill at UFC 300?",
        "Tell me about Alex Pereira's recorded career stats.",
        "Summarize UFC 300.",
        "How many recorded wins does Khamzat Chimaev have?",
        "Which fighters and fights are mentioned in the knowledge base?",
    ],
)

# share=True creates a temporary public URL. Do not use it for private/sensitive data.
demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9b2d4b709bc7d8e97a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [24]:
# Cell 17 — Cleanup (run only after you finish using the agent)
# The app can be stopped from the Gradio interface/runtime. Close the read-only DB connection when finished.
# conn.close()
print("Notebook 5 setup is complete. Keep the runtime running while you use the Gradio chat.")


Notebook 5 setup is complete. Keep the runtime running while you use the Gradio chat.
